# Lab 1.2 — Training and Evaluating a First ML Model
**Module I · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_labs/blob/main/module-1-ml/lab1_2_first_ml_model.ipynb)

---

## What you will do
In this lab you will go through the full supervised-learning workflow:
1. **Prepare** the data: encode categoricals, split into train / val / test.
2. **Train** a Logistic Regression model.
3. **Evaluate** it — first naïvely (accuracy), then properly (confusion matrix, precision, recall, F1).
4. `[Extension]` Compare with a Decision Tree and diagnose overfitting.

## Prerequisites
You should have completed **Lab 1.1** (or at least be familiar with the Telco Churn dataset).

**Estimated time:** 60–90 min (core) + 20–30 min (extensions)

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("labs_assignment").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/DanielFPerez/llm-gnns-course_labs.git",
             "labs_assignment"],
            check=True,
        )
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         "labs_assignment/environment/requirements.txt"],
        check=True, capture_output=True, text=True,
    )
    sys.path.insert(0, "labs_assignment")
    if "Successfully installed" in result.stdout:
        print("Packages installed. Restarting runtime — re-run all cells after reconnecting.")
        import os; os.kill(os.getpid(), 9)
else:
    # Running locally from inside the module folder — go one level up
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
)

from utils import (
    load_telco_churn,
    plot_class_distribution,
    plot_confusion_matrix,
    plot_training_curves,
    plot_feature_importance,
)
from utils.checks import check_dataframe, check_split, check_model

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110
print("Imports OK.")

---
## 1 · Data preparation

The cleaning steps from Lab 1.1 are provided below — no exercises here. Run this cell to get a clean DataFrame ready for modelling.

In [ ]:
df_raw = load_telco_churn()

df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID"])   # ID column carries no signal

print(f"Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Churn rate: {df['Churn'].mean():.1%}")

---
## 2 · Feature encoding

Most ML models require **numerical inputs**. The Telco dataset has many text columns (`gender`, `Contract`, `PaymentMethod`, …). We need to convert them to numbers.

### Exercise 2.1 `[Core]` — One-hot encode the categorical columns

Use `pd.get_dummies()` on the DataFrame (excluding the target column `Churn`). Store the result in `df_encoded`, making sure `Churn` is still a column in it.

**Tip:** `pd.get_dummies(df, drop_first=True)` automatically handles all `object` columns and drops one dummy per category to avoid collinearity.

In [ ]:
# YOUR CODE HERE
# Hint: Use pd.get_dummies() on the DataFrame (excluding the target column Churn). Store the result in df_encoded, making sur...
raise NotImplementedError("Complete this exercise")

In [ ]:
check_dataframe(
    "2.1", df_encoded,
    min_rows=7000,
    required_cols=["Churn", "tenure", "MonthlyCharges"],
)

---
## 3 · Train / Validation / Test split

We split the data **three ways**:
- **Train (70%):** the model learns from this.
- **Validation (15%):** we tune and compare models here, without touching the test set.
- **Test (15%):** held out until the very end — our unbiased estimate of real-world performance.

We use `stratify=y` to preserve the ~73/27 class ratio in all three sets.

### Exercise 2.2 `[Core]` — Split the data

Create `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test` with a 70/15/15 split. Use `random_state=42` and `stratify=y` throughout.

**Tip:** Do it in two steps — first split off the test set, then split the remaining 85% into train and val.

In [ ]:
# YOUR CODE HERE
# Hint: Create X_train, X_val, X_test, y_train, y_val, y_test with a 70/15/15 split. Use random_state=42 and stratify=y throu...
raise NotImplementedError("Complete this exercise")

In [ ]:
check_split("2.2", X_train, X_val, X_test, y_train, y_val, y_test)

---
## 4 · Training a Logistic Regression model

**Logistic Regression** is a strong, interpretable baseline for binary classification. Despite its name, it is a *classification* algorithm: it outputs the probability that a sample belongs to class 1 (Churn) and applies a threshold (default: 0.5).

Because features have very different scales (`tenure` goes 0–72, `MonthlyCharges` goes 18–119), we first **standardise** them: subtract the mean and divide by the standard deviation. This is required for Logistic Regression to converge well.

> **Important:** Fit the scaler **only on the training set**, then apply it to val and test. Fitting on all data would let information from the future leak into training — a form of data leakage.

### Exercise 2.3 `[Core]` — Scale features and train the model

1. Create a `StandardScaler` and fit it on `X_train`. Transform `X_train`, `X_val`, and `X_test`.
2. Create a `LogisticRegression(max_iter=500, random_state=42)` and train it on the scaled training data.
3. Store the fitted model in a variable called `lr`.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Create a StandardScaler and fit it on X_train. Transform X_train, X_val, and X_test.
raise NotImplementedError("Complete this exercise")

---
## 5 · Evaluation

Now we measure how well the model performs. We will evaluate it on the **validation set** (never the test set — that is saved for the end).

### Exercise 2.4 `[Core]` — Accuracy: the misleading metric

Compute the accuracy of `lr` on the validation set. Then compute what accuracy a **dummy classifier** that always predicts the majority class (No Churn = 0) would achieve.

Do the two numbers feel close? Why is accuracy alone not enough here?

In [ ]:
# YOUR CODE HERE
# Hint: Compute the accuracy of lr on the validation set. Then compute what accuracy a dummy classifier that always predicts ...
raise NotImplementedError("Complete this exercise")

### Exercise 2.5 `[Core]` — Confusion matrix

Plot the confusion matrix for the Logistic Regression predictions on the validation set. Use the `plot_confusion_matrix` helper from `utils`.

> **Reminder (Lecture, Slide 18):**
> - **True Positive (TP):** predicted Churn, actually Churn.
> - **True Negative (TN):** predicted No Churn, actually No Churn.
> - **False Positive (FP):** predicted Churn, actually No Churn (false alarm).
> - **False Negative (FN):** predicted No Churn, actually Churn (missed churner — costly!).

In [ ]:
# YOUR CODE HERE
# Hint: Plot the confusion matrix for the Logistic Regression predictions on the validation set. Use the plot_confusion_matri...
raise NotImplementedError("Complete this exercise")

> **What to notice:** The model catches a decent fraction of churners (TP) but also produces many false negatives (FN) — actual churners predicted as staying. In a real deployment, every FN is a lost customer. This is why recall matters more than accuracy here.

### Exercise 2.6 `[Core]` — Precision, Recall, and F1

Print the full `classification_report` for the Logistic Regression on the validation set. Identify:
- The **recall** for class 1 (Churn) — what fraction of actual churners did we catch?
- The **F1-score** for class 1 — the harmonic mean of precision and recall.

In [ ]:
# YOUR CODE HERE
# Hint: Print the full classification_report for the Logistic Regression on the validation set. Identify:
raise NotImplementedError("Complete this exercise")

In [ ]:
check_model("2.6", lr, X_val_sc, y_val, min_f1=0.4)

> **Interpretation guide:**
> - **Precision (Churn):** of everyone we flagged as likely to churn, what fraction actually did? Low precision → many false alarms (wasted retention campaigns).
> - **Recall (Churn):** of everyone who actually churned, what fraction did we catch? Low recall → missed churners (lost revenue).
> - **F1 (Churn):** the single number that balances both. Use this to compare models.

---
## 6 · `[Extension]` Comparing models: Decision Tree

Logistic Regression is linear. A **Decision Tree** can capture non-linear patterns (e.g., "high charges AND short tenure → likely to churn"). Let's compare.

### Exercise 2.7 `[Extension]` — Train a Decision Tree and compare F1

1. Train a `DecisionTreeClassifier(random_state=42)` on the *unscaled* training data (decision trees don't need scaling).
2. Print the classification report on the validation set.
3. Compare the **Churn F1-score** with Logistic Regression. Which is better? Why might that be?

In [ ]:
# YOUR CODE HERE
# Hint: 1. Train a DecisionTreeClassifier(random_state=42) on the unscaled training data (decision trees don't need scaling).
raise NotImplementedError("Complete this exercise")

> **Common finding:** The unconstrained Decision Tree often achieves near-perfect training accuracy but lower validation performance than Logistic Regression. This is **overfitting** — the tree memorises the training data, including its noise, and fails to generalise. We diagnose this properly in Exercise 2.8.

### Exercise 2.8 `[Extension]` — Diagnosing overfitting with learning curves

Train the Decision Tree on increasingly large subsets of the training data and record both **train F1** and **val F1** at each size. A large gap between the two curves signals overfitting.

Then try limiting the tree depth (`max_depth=5`) and see if the gap closes.

In [ ]:
# YOUR CODE HERE
# Hint: Train the Decision Tree on increasingly large subsets of the training data and record both train F1 and val F1 at eac...
raise NotImplementedError("Complete this exercise")

In [ ]:
# Now with max_depth=5 to reduce overfitting
train_f1s_p, val_f1s_p = [], []

for frac in train_sizes:
    n = max(50, int(frac * len(X_train)))
    X_sub, y_sub = X_train.iloc[:n], y_train.iloc[:n]

    m = DecisionTreeClassifier(max_depth=5, random_state=42)
    m.fit(X_sub, y_sub)

    train_f1s_p.append(f1_score(y_sub, m.predict(X_sub), pos_label=1))
    val_f1s_p.append(f1_score(y_val, m.predict(X_val), pos_label=1))

plot_training_curves(
    train_f1s_p, val_f1s_p,
    metric="Churn F1",
    title="Learning curve — Decision Tree (max_depth=5)",
)
plt.show()

> **Key takeaway:** With `max_depth=5` the train/val gap shrinks significantly — the curves converge. This is **regularisation**: constraining the model's complexity so it generalises better. We have just manually implemented the core idea behind model selection.

---
## 7 · `[Extension]` Feature importance

### Exercise 2.9 `[Extension]` — Which features does the model rely on most?

Extract the feature importances from the Decision Tree (attribute: `.feature_importances_`) and visualise the top 15 using `plot_feature_importance` from `utils`. Do the top features match the correlations you found in Lab 1.1?

In [ ]:
# YOUR CODE HERE
# Hint: Extract the feature importances from the Decision Tree (attribute: .feature_importances_) and visualise the top 15 us...
raise NotImplementedError("Complete this exercise")

---
## 8 · Final evaluation on the test set

We have been evaluating everything on the **validation set**. Now that we have chosen our best model (Logistic Regression, based on the validation F1), we run it exactly **once** on the test set to get an unbiased estimate of real-world performance.

> Do this only once. If you look at the test set repeatedly and adjust the model based on it, you are effectively overfitting to the test set.

In [ ]:
y_pred_test = lr.predict(X_test_sc)

print("=== Final evaluation — Logistic Regression on TEST set ===")
print(classification_report(y_pred_test, y_test, target_names=["No Churn", "Churn"]))

cm_test = confusion_matrix(y_test, y_pred_test)
plot_confusion_matrix(cm_test, class_names=["No Churn", "Churn"], title="Confusion matrix — Test set");

---
## Summary

| Step | What we did | Key lesson |
|---|---|---|
| Feature encoding | `pd.get_dummies` | ML needs numbers; text must be encoded |
| Train/val/test split | 70/15/15, stratified | Never touch the test set until the end |
| Feature scaling | `StandardScaler` (fit on train only) | Prevents data leakage; required for LR |
| Accuracy | ~81% — looked good | 73% is free from the majority class |
| Confusion matrix | Revealed many FNs | Missed churners are a real business cost |
| Precision / Recall / F1 | Proper evaluation | F1 balances both sides of the trade-off |
| Decision Tree `[Ext]` | Higher train, lower val accuracy | Classic overfitting — cured by `max_depth` |
| Feature importance `[Ext]` | `Contract`, `tenure`, `MonthlyCharges` dominate | Matches the EDA intuition from Lab 1.1 |

---

## Connection to the final project

You now have the complete Module I workflow: load → clean → encode → split → train → evaluate. In the final project, you will apply this same workflow to a dataset of your choice. Starting in **Module II**, we will replace Logistic Regression with an LLM and augment it with a retrieval layer — and eventually (Module IV) with a Graph Neural Network.

> **The ML fundamentals you practised today are not a detour — they are the foundation everything else builds on.**